# 02 — Profitability by Region
Total sales, profit, and margin compared across geographic regions.


In [ ]:
import os, sys
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

os.environ['JAVA_HOME']             = '/usr/local/java'
os.environ['SPARK_HOME']            = '/usr/local/spark'
os.environ['HADOOP_CONF_DIR']       = '/usr/local/hadoop/etc/hadoop'
os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

try:
    spark.stop()
except:
    pass

# Local mode — no Hive/hive-metastore dependency
spark = (SparkSession.builder
    .appName("Superstore Analytics")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# ── Load CSVs and register temp views ─────────────────────────────────────────
DATA = "/usr/local/hadoop/etc/hadoop/assessment-2"

def load(filename, renames):
    df = spark.read.option("header", "true").option("inferSchema", "true") \
             .csv(f"{DATA}/{filename}")
    for old, new in renames.items():
        df = df.withColumnRenamed(old, new)
    return df

customers  = load("customers.csv",  {"Customer ID": "customer_id",
                                      "Customer Name": "customer_name",
                                      "Segment": "segment"})
orders     = load("orders.csv",     {"Order ID": "order_id",
                                      "Order Date": "order_date",
                                      "Ship Date": "ship_date",
                                      "Ship Mode": "ship_mode",
                                      "Customer ID": "customer_id",
                                      "Postal Code": "postal_code"})
order_items = load("order_items.csv", {"Row ID": "row_id",
                                        "Order ID": "order_id",
                                        "Product ID": "product_id",
                                        "Sales": "sales",
                                        "Quantity": "quantity",
                                        "Discount": "discount",
                                        "Profit": "profit"})
products   = load("products.csv",   {"Product ID": "product_id",
                                      "Product Name": "product_name",
                                      "Category": "category",
                                      "Sub-Category": "sub_category"})
locations  = load("locations.csv",  {"Postal Code": "postal_code",
                                      "City": "city",
                                      "State": "state",
                                      "Country": "country",
                                      "Region": "region"})

customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
products.createOrReplaceTempView("products")
locations.createOrReplaceTempView("locations")

print("Spark", spark.version, "ready — all tables loaded.")
spark.sql("SHOW TABLES").show()

# ── Global chart style ─────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"


In [ ]:
# ── Profitability by Region ───────────────────────────────────────────────────
region = spark.sql("""
    SELECT l.region,
           ROUND(SUM(oi.sales),  2)                    AS sales,
           ROUND(SUM(oi.profit), 2)                    AS profit,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100, 2) AS margin_pct
    FROM orders o
    JOIN order_items oi ON o.order_id    = oi.order_id
    JOIN locations   l  ON o.postal_code = l.postal_code
    GROUP BY l.region
    ORDER BY profit DESC
""").toPandas()

colors = [PALETTE[1] if v >= 0 else PALETTE[3] for v in region["profit"]]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(region["region"], region["profit"],
              color=colors, edgecolor="white", zorder=3)
for bar, mg in zip(bars, region["margin_pct"]):
    h      = bar.get_height()
    offset = max(abs(h) * 0.03, 500)
    sign   = 1 if h >= 0 else -1
    ax.text(bar.get_x() + bar.get_width()/2,
            h + sign * offset,
            f"{fmt_usd(h)}\n({mg}% margin)",
            ha="center", va="bottom" if h >= 0 else "top",
            fontsize=9, fontweight="bold")
ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.set_title("Total Profit by Region")
ax.set_ylabel("Profit (USD)")
ax.set_xlabel("Region")
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()


In [ ]:
spark.stop()
print("Spark stopped.")
